# Transform Constructors Data

1. Read bronze `constructors` table
1. Keep only the columns required for analytics (Drop `url` column)
1. Standardise column names using snake_case (`constructorId` → `constructor_id`)
1. Rename columns to make them more meaningful (`name` → `constructor_name`)
1. Remove duplicate records
1. Transform values of column `nationality` to Title Case
1. Write the transformed data to silver `constructors` table



#### Entity Relationship Diagram - Formula1 Bronze Schema

![Formula1 Raw Data.png](../../z-course-images/formula1-raw-data-erd.png "Formula1 Raw Data.png")

In [0]:
dbutils.widgets.text("p_batch_id", "")
v_batch_id = dbutils.widgets.get("p_batch_id")  

In [0]:
%run ../00-common/01.environment-config

In [0]:
%run ../00-common/03.silver-helpers

In [0]:
bronze_table = f"{catalog_name}.{bronze_schema}.constructors"
silver_table = f"{catalog_name}.{silver_schema}.constructors"

In [0]:
from pyspark.sql import functions as F

#### Step 1 - Read bronze `constructors` table

In [0]:
constructors_df = (
    spark.table(bronze_table)
         .filter((F.col("batch_id") == v_batch_id))
)

#### Step 2 - Keep only the columns required for analytics (Drop url column)

In [0]:
constructors_dropped_df = constructors_df.drop("url")

#### Step 3 & 4 - Standardise Column Names
- Standardise column names using snake_case (`constructorId` → `constructor_id`)
- Rename columns to make them more meaningful (`name` → `constructor_name`)

In [0]:
constructors_renamed_df = (
    constructors_dropped_df
        .withColumnsRenamed({
            "constructorId": "constructor_id",
            "name": "constructor_name"
        })
)

In [0]:
display(constructors_renamed_df)

#### Step 5 - Remove duplicate records

In [0]:
constructors_distinct_df = constructors_renamed_df.dropDuplicates(["constructor_id"])

In [0]:
display(constructors_distinct_df)

#### Step 6 - Transform values of column `nationality` to Title Case

In [0]:
constructors_final_df = (
    constructors_distinct_df
        .withColumn('nationality', F.initcap(F.col("nationality")))
)

In [0]:
display(constructors_final_df)

#### Step 7 - Write the transformed data to silver `constructors` table

In [0]:
write_to_silver(
    input_df=constructors_final_df,
    target_table=silver_table,
    merge_condition="t.constructor_id = s.constructor_id",
    columns_to_update=[
        "constructor_name",
        "nationality",
        "ingestion_timestamp",
        "source_file",
        "batch_id"
    ]
)

In [0]:
display(spark.table(silver_table))